# Instagram two-agent pipeline (Agent1 unchanged) + Tavily evidence + improved Agent2

- Agent1: splits text into **Discriminative Text** and **Residual Text** (kept identical to your previous logic).
- Tavily: searches for **contact/company legitimacy + scam reports** for discriminative samples.
- Agent2: makes final label using **(a) agent1 output + (b) Tavily evidence**.

> **Label convention used here (based on your `instagram_final_results` errors):**
> - **1 = benign / normal profile**
> - **0 = suspicious / potential scam / unverifiable contact**

Set environment variables before running:
- `OPENAI_API_KEY`, `OPENAI_BASE_URL` (optional), `TAVILY_API_KEY`


In [ ]:

import os
import re
import time
import json
from typing import List, Dict, Any, Tuple

import requests
import pandas as pd
from openai import OpenAI


# 0) Keys / Clients

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "Input your key")
OPENAI_BASE_URL = os.getenv("OPENAI_API_BASE", os.getenv("OPENAI_BASE_URL", "Input your URL"))

# Tavily
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "Input your key")
TAVILY_ENDPOINT = os.getenv("TAVILY_ENDPOINT", "Input your URL")

if not OPENAI_API_KEY:
    raise RuntimeError("Missing OPENAI_API_KEY env var")

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)


def retry_with_backoff(func, *args, retries=10, delay=2, **kwargs):
    """Simple exponential backoff retry."""
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(delay)
                delay *= 2
            else:
                raise last_err


def response_API(user_prompt: str, model: str = "gpt-4o-mini", temperature: float = 0.0) -> str:
    """Call LLM (single-user message, compatible with your original code)."""

    def api_call():
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": user_prompt}],
            temperature=temperature,
        )
        return resp.choices[0].message.content

    return retry_with_backoff(api_call, retries=8, delay=2)



# 1) Agent1 prompt (UNCHANGED)

AGENT1_PROMPT = r"""
You are a text splitter/classifier.
TASK
Split the provided input text into TWO buckets:
1) Discriminative Text
2) Residual Text
NON-NEGOTIABLE RULES (VERY IMPORTANT)
- Do NOT add any commentary, explanation, bullets, or extra headings.
- Do NOT paraphrase, summarize, translate, rewrite, or normalize.
- Do NOT delete anything: every line from the input MUST appear exactly once
  in either Discriminative Text or Residual Text.
- Preserve the original order of lines exactly as in the input.
- Keep emojis and special characters exactly as-is.
- If you are unsure, put it into Residual Text (but still do not delete it).
HARD DISCRIMINATIVE RULES (must go to Discriminative Text)
If a line contains ANY of the following, it MUST be Discriminative Text:
- '@' (email address or social handle, e.g., name@gmail.com, @username)
- '#' (hashtag/topic tag, e.g., #SEO, #London, #Trending, #Crypto)
- A URL or domain (http, https, www, .com, .co, .uk, .net, linktr.ee, beacons.ai, etc.)
- Contact / collaboration / business signals:
  "DM", "inquiries", "enquiries", "contact", "email", "collab", "collaborate",
  "booking", "bookings", "manager", "managed by", "agent", "PR", "press",
  "WhatsApp", "WeChat", "Line", "Telegram", "Kakao", "Snap", "Discord"
- Payment/account/channel identifiers:
  bank names, card brands, PayPal, Venmo, CashApp, Alipay, WeChat Pay, IBAN,
  sort code, account number, routing number, wallet address, crypto address
- Any explicit call-to-action to message/click/link/pay/subscribe.
WHAT GOES TO RESIDUAL
Everything else that does not match the hard discriminative rules.
OUTPUT FORMAT (MUST MATCH EXACTLY)
Discriminative Text:
<copy lines here exactly>
Residual Text:
<copy remaining lines here exactly>
Remember:
- First line must be exactly: Discriminative Text:
- Then later exactly: Residual Text:
- Output ONLY these two sections, nothing else.
""".strip()



def agent1_process(original_text) -> Dict[str, str]:
    if not isinstance(original_text, str):
        original_text = ""

    full_prompt = AGENT1_PROMPT + "\n\n" + original_text


    try:
        response = response_API(full_prompt)
    except Exception as e:
        print("Agent1 API failed:", e)
        response = ""

    if not response.strip():
        return {"Discriminative Text": "", "Residual Text": original_text or ""}

    disc_lines, resid_lines = [], []
    current_section = None

    for line in response.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("Discriminative Text"):
            current_section = "D"
            continue
        if line.startswith("Residual Text"):
            current_section = "R"
            continue
        if current_section == "D":
            disc_lines.append(line)
        elif current_section == "R":
            resid_lines.append(line)

    return {
        "Discriminative Text": "\n".join(disc_lines),
        "Residual Text": "\n".join(resid_lines),
    }


def run_agent1_and_save_csv(input_csv: str, output_csv: str, sleep_s: float = 0.5):
    df = pd.read_csv(input_csv)

    disc_col, resid_col = [], []
    for idx, row in df.iterrows():
        text = row.get("text", "")
        result = agent1_process(text)
        disc_col.append(result["Discriminative Text"])
        resid_col.append(result["Residual Text"])

        if idx % 10 == 0:
            print(f"Agent1 processed {idx} samples")
        time.sleep(sleep_s)

    df["Discriminative Text"] = disc_col
    df["Residual Text"] = resid_col
    df.to_csv(output_csv, index=False)
    print("Agent1 CSV saved to:", output_csv)


# 2) Tavily search (pattern copied from your combined_script style)


def tavily_search(query: str, retries: int = 3, delay: int = 2, max_results: int = 2, search_depth: str = "basic") -> Dict[str, Any]:
    if not TAVILY_API_KEY:
        raise RuntimeError("Missing TAVILY_API_KEY env var")

    headers = {"Content-Type": "application/json"}
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": max_results,
        "search_depth": search_depth,
        "include_answer": False,
        "include_raw_content": False,
    }

    def api_call():
        resp = requests.post(TAVILY_ENDPOINT, json=payload, headers=headers, timeout=30)
        resp.raise_for_status()
        return resp.json()

    return retry_with_backoff(api_call, retries=retries, delay=delay)


EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE_RE = re.compile(r"(?:(?:\+|00)\d{1,3}[\s-]?)?(?:\(?\d{2,4}\)?[\s-]?)?\d{3,4}[\s-]?\d{3,4}")
HANDLE_RE = re.compile(r"(?<!\w)@[A-Za-z0-9._]{2,30}")
URL_RE = re.compile(r"https?://\S+|\b(?:[A-Za-z0-9-]+\.)+[A-Za-z]{2,}\b")


def extract_search_targets(discriminative_text: str) -> List[str]:
    """Extract emails/phones/handles/domains to look up."""
    disc = discriminative_text or ""
    targets = []

    targets += EMAIL_RE.findall(disc)

    # Phones: filter out very short matches that are likely dates or counts
    for m in PHONE_RE.findall(disc):
        digits = re.sub(r"\D", "", m)
        if len(digits) >= 9 and len(digits) <= 15:
            targets.append(m.strip())

    targets += HANDLE_RE.findall(disc)

    # domains/urls
    for m in URL_RE.findall(disc):
        m = m.strip().strip(".,;:!()[]{}<>")
        if m and m not in targets:
            targets.append(m)

    # de-dup preserve order
    seen = set()
    out = []
    for t in targets:
        if t.lower() not in seen:
            seen.add(t.lower())
            out.append(t)
    return out[:6]  # cap


def tavily_evidence_for_row(row: pd.Series) -> Dict[str, Any]:
    """
    Only search when Agent1 produced valid discriminative text.
    If discriminative text is empty / NaN / meaningless → skip Tavily.
    """

    disc = row.get("Discriminative Text", None)

    if disc is None:
        return {"targets": [], "queries": [], "results": []}

    if not isinstance(disc, str):
        return {"targets": [], "queries": [], "results": []}

    disc = disc.strip()
    if disc == "" or disc.lower() == "nan":
        return {"targets": [], "queries": [], "results": []}

    targets = extract_search_targets(disc)
    if not targets:
        return {"targets": [], "queries": [], "results": []}

    queries = []
    results_compact = []

    for t in targets:
        q = f"{t} scam OR fraud OR complaint OR review"
        queries.append(q)

        try:
            res = tavily_search(q, max_results=2, search_depth="basic")
        except Exception as e:
            res = {"error": str(e), "results": []}

        compact_items = []
        for item in (res.get("results") or [])[:2]:
            compact_items.append({
                "title": item.get("title", ""),
                "url": item.get("url", ""),
                "content": item.get("content", "")[:280],
                "score": item.get("score", None),
            })

        results_compact.append({
            "target": t,
            "query": q,
            "items": compact_items
        })

        time.sleep(0.2)

    return {
        "targets": targets,
        "queries": queries,
        "results": results_compact
    }



def add_tavily_evidence_column(agent1_csv: str, output_csv: str, sleep_s: float = 0.2):
    df = pd.read_csv(agent1_csv)

    evidences = []
    for idx, row in df.iterrows():
        ev = tavily_evidence_for_row(row)
        evidences.append(json.dumps(ev, ensure_ascii=False))
        if idx % 10 == 0:
            print(f"Tavily processed {idx} samples")
        time.sleep(sleep_s)

    df["tavily_evidence"] = evidences
    df.to_csv(output_csv, index=False)
    print("Agent1+Tavily CSV saved to:", output_csv)



# 3) Improved Agent2 prompt (fix label direction + reduce false positives)


AGENT2_PROMPT = r"""
You are Agent2, a strict binary classifier.
Return ONLY one digit: 0 or 1.

You are given:
- text: full Instagram bio
- Discriminative Text: extracted high-signal cues (may be empty)
- Residual Text: remaining context (low priority)
- tavily_evidence: JSON (may be empty)

GOAL:
Match the dataset labels. In this dataset, many bios that look like creators/influencers
are still label=0. Therefore, you must NOT output 1 based on weak “creator vibes”.


A) PRIORITY & NORMALIZATION

1) Discriminative Text > text > Residual Text.
2) If Discriminative Text is empty, do NOT assume positive. Treat as unknown.
3) tavily_evidence empty is NEUTRAL (not positive, not negative).
4) Ignore corrupted emoji/encoding artifacts like "ðŸ", "ğŸ" etc.


B) CORE DECISION: SCORE + THRESHOLD

Compute a score using ONLY explicit cues in the text.
Default output is 0 unless the score reaches the threshold.

Start score = 0.

Add STRONG POSITIVE signals (+3 each):
+3  Mentions a recognized institution/platform/media network explicitly:
    Examples: "AM 640", "iHeartRadio", "HGTV", "NBC", "radio", "TV presenter"
+3  Clearly indicates an official organization/brand/service venue with operations:
    Examples: salon + booking schedule (W-2-F / appointments / books closed),
    business address + official brand handle, store/clinic/studio name + services.
+3  Clear official topic authority brand like "@SCIENCE" + inquiries email.

Add MODERATE POSITIVE signals (+2 each):
+2  Explicit "Business Inquiries:" / "Press:" / "Media:" + contact email
+2  Two or more consistent external handles pointing to a brand ecosystem
    (e.g., multiple linked brand accounts + role like Founder/Owner)
+2  tavily_evidence contains matching results that align with the bio identity
    (must be consistent; do not assume if results are unclear)

Add WEAK signals (+1 each) [NEVER sufficient alone]:
+1  Has an email
+1  Has "collab", "DM for collabs"
+1  Has "content creator", "influencer", "blogger"
+1  Has TikTok/YouTube mention
+1  Has generic profession label (photographer, stylist, coach) without external verification

Add STRONG NEGATIVE signals (-3 each):
-3  Pure diary / personal vibe only, no operational or institutional cues:
    examples: "visual diary", "my life", random quotes only
-3  Generic lifestyle list only (travel | fashion | beauty | mindset) with no verifiable anchor
-3  Text is mostly hashtags/emojis with no concrete role/entity

Add MODERATE NEGATIVE signals (-2 each):
-2  Overly generic “creator template” bios:
    "lifestyle | travel | fashion | mindset | growth" + DM/email only
-2  Only location + personal descriptors (mom/wife/dog lover) without institution/entity
-2  Discriminative Text contains only social handles or hashtags, no real-world anchor

C) OUTPUT THRESHOLD

If score >= 4  -> output 1
Else           -> output 0

IMPORTANT SAFETY GUARDRAILS:
- Never output 1 if the ONLY evidence is: email + "collab" + "lifestyle/blogger".
  (This pattern is frequently label=0 in this dataset.)
- If uncertain, output 0 (because false positives dominate your current errors).
- Return ONLY 0 or 1. No explanation.



""".strip()


def agent2_judge_from_row(row: pd.Series, sleep_s: float = 0.0) -> int:
    disc = row.get("Discriminative Text", "")
    resid = row.get("Residual Text", "")
    tav = row.get("tavily_evidence", "")

    # If Agent1 found nothing discriminative, default to benign.
    if not isinstance(disc, str) or not disc.strip():
        return 1

    user_input = f"""
Discriminative Text:\n{disc}

Residual Text:\n{resid}

Tavily evidence JSON:\n{tav}
""".strip()

    resp = response_API(AGENT2_PROMPT + "\n\n" + user_input)
    time.sleep(sleep_s)

    try:
        val = int(resp.strip())
        return 1 if val == 1 else 0
    except Exception:
        return 1


def run_agent2_and_save_csv(agent1_tavily_csv: str, output_csv: str, sleep_s: float = 0.5):
    df = pd.read_csv(agent1_tavily_csv)

    preds = []
    for idx, row in df.iterrows():
        pred = agent2_judge_from_row(row)
        preds.append(pred)
        if idx % 10 == 0:
            print(f"Agent2 processed {idx} samples")
        time.sleep(sleep_s)

    df["agent2_pred"] = preds
    df.to_csv(output_csv, index=False)
    print("Final CSV saved to:", output_csv)



# 4) Main (Colab-friendly paths)

if __name__ == "__main__":
    # Update these paths as needed
    input_csv = "/content/instagram_samples.csv"
    after_a1 = "/content/instagram_after_agent1.csv"
    after_tav = "/content/instagram_after_tavily.csv"
    final_out = "/content/instagram_final_results.csv"

    run_agent1_and_save_csv(input_csv=input_csv, output_csv=after_a1)
    add_tavily_evidence_column(agent1_csv=after_a1, output_csv=after_tav)
    run_agent2_and_save_csv(agent1_tavily_csv=after_tav, output_csv=final_out)


Agent1 processed 0 samples
Agent1 processed 10 samples
Agent1 processed 20 samples
Agent1 processed 30 samples
Agent1 processed 40 samples
Agent1 CSV saved to: /content/instagram_after_agent1.csv
Tavily processed 0 samples
Tavily processed 10 samples
Tavily processed 20 samples
Tavily processed 30 samples
Tavily processed 40 samples
Agent1+Tavily CSV saved to: /content/instagram_after_tavily.csv
Agent2 processed 0 samples
Agent2 processed 10 samples
Agent2 processed 20 samples
Agent2 processed 30 samples
Agent2 processed 40 samples
Final CSV saved to: /content/instagram_final_results.csv


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
df_result = pd.read_csv("/content/instagram_final_results.csv")
y_true = df_result["label"]
y_pred = df_result["agent2_pred"]

# 只保留 y_true 非空且 y_pred 非空的行
mask = y_true.notna() & y_pred.notna()

y_true_clean = y_true[mask]
y_pred_clean = y_pred[mask]

acc = accuracy_score(y_true_clean, y_pred_clean)
p   = precision_score(y_true_clean, y_pred_clean, average="binary")
r   = recall_score(y_true_clean, y_pred_clean, average="binary")
f1  = f1_score(y_true_clean, y_pred_clean, average="binary")

print("Used rows:", mask.sum(), "/", len(df_result))
print("Accuracy:", acc, "Precision:", p, "Recall:", r, "F1:", f1)



Used rows: 50 / 50
Accuracy: 0.68 Precision: 0.42857142857142855 Recall: 0.42857142857142855 F1: 0.42857142857142855
